
The standard "treacherous" probability problem is the famous <a href="https://en.wikipedia.org/wiki/">Monty Hall problem</a>. Some of the issue is translating a word problem into math terms, and a lot of it is unfamiliarity with probabilistic reasoning (most majors don't spend a lot of time on this). I think I have found one that is even thornier: "Boy Born on Tuesday."

I really like Peter Winkler discussion of the *seemingly* simple probability problem "Boy Born on Tuesday" from his <i>Mathematical Puzzles</i> CRC Press, 2024, pp. xix, xcii, 113-114.

Instead of stealing his thunder I'll merely present a simplified version in terms of coin flips. I'll have Python enumerate all the cases to get the answer and leave the reasoning to interested readers.

I restate the problem as:

<blockquote>
A person flipped two independent fair (50/50 chance of heads or tails) each on a random independent uniformly chosen day of the week (7 days, Sunday through Saturday, duplicate days allowed, chosen independent of any coin flip results). If they truthfully report "I had at least one heads result on a Tuesday", then what is the probability that both coin flips came up heads?
</blockquote>

This is to emphasize this really is a probabilty problem, not a subtle biology issue (such as rates of identical twins or such).

It turns the statement is leaking information about the flip. For example it eliminates cases where they didn't flip on any Tuesdays, and the case where they saw two tails results on a Tuesday. The leaked information means the probability of both coin flips coming up heads (conditioned on the leaded information) is not any of the simple guesses such as 0, 1/4, 1/3, 1/2, 2/3, 3/4, or 1.

This problem is considered hard by most on first exposure. It isn't that the combinatorics that is hard here, it is the usual trap of trusting intuition over calculation when reasoning about probabilities. In this case intuition tends to be weak (the problem is a bit artificial, therefore a bit unfamiliar) and the obvious calculation is very non-informative (unappetizing case enumeration).

Below I work some warm-up problems directly in Python (such as the famous "at least one of the two is heads, what is then the probability both are heads") and then the "Boy Born on Tuesday" problem. For more discussion I recommend Peter Winkler "Boy Born on Tuesday" from his <i>Mathematical Puzzles</i> CRC Press, 2024, pp. xix, xcii, 113-114 (heck, I in general very strongly recommend this wonderful book).


In [1]:
from itertools import product
import numpy as np
import pandas as pd
import sympy as sp

Build data frame of all equally likely possible outcomes.

In [2]:
coin_sides = ('heads', 'tails')
days_of_week = ('Sunday', 'Monday', 'Tuesday', 'Wedndesday', 'Thursday', 'Friday', 'Saturday')

exact_frame = pd.DataFrame(
    product(coin_sides, days_of_week, coin_sides, days_of_week),
    columns=[
        'flip[1] side',
        'flip[1] birth day of week',
        'flip[2] side',
        'flip[2] birth day of week',
    ],
)

exact_frame

,flip[1] side,flip[1] birth day of week,flip[2] side,flip[2] birth day of week
0,heads,Sunday,heads,Sunday
1,heads,Sunday,heads,Monday
2,heads,Sunday,heads,Tuesday
3,heads,Sunday,heads,Wedndesday
4,heads,Sunday,heads,Thursday
...,...,...,...,...
191,tails,Saturday,tails,Tuesday
192,tails,Saturday,tails,Wedndesday
193,tails,Saturday,tails,Thursday
194,tails,Saturday,tails,Friday


Mark the data frame with observations.

In [3]:

exact_frame['at least 1 heads'] = (
    exact_frame['flip[1] side'] == 'heads'
) | (
    exact_frame['flip[2] side'] == 'heads'
)

exact_frame['both heads'] = (
    exact_frame['flip[1] side'] == 'heads'
) & (
    exact_frame['flip[2] side'] == 'heads'
)

exact_frame['at least one heads on Tuesday'] = (
    (exact_frame['flip[1] side'] == 'heads') & (exact_frame['flip[1] birth day of week'] == 'Tuesday')
) | (
    (exact_frame['flip[2] side'] == 'heads') & (exact_frame['flip[2] birth day of week'] == 'Tuesday')
)

exact_frame

,flip[1] side,flip[1] birth day of week,flip[2] side,flip[2] birth day of week,at least 1 heads,both heads,at least one heads on Tuesday
0,heads,Sunday,heads,Sunday,True,True,False
1,heads,Sunday,heads,Monday,True,True,False
2,heads,Sunday,heads,Tuesday,True,True,True
3,heads,Sunday,heads,Wedndesday,True,True,False
4,heads,Sunday,heads,Thursday,True,True,False
...,...,...,...,...,...,...,...
191,tails,Saturday,tails,Tuesday,False,False,False
192,tails,Saturday,tails,Wedndesday,False,False,False
193,tails,Saturday,tails,Thursday,False,False,False
194,tails,Saturday,tails,Friday,False,False,False


Try some calculations.

First the unconditional probability both coin flips come up heads.

In [4]:
p_both_heads = float(np.mean(exact_frame['both heads']))

p_both_heads

0.25

In [5]:
assert np.abs(p_both_heads - 1 / 4) < 1e-8

Now the probility both coin flips come up heads given that the first flip came up heads.

In [6]:
p_both_heads_given_first_is_heads = float(np.mean(exact_frame.loc[exact_frame['flip[1] side'] == 'heads', 'both heads']))

p_both_heads_given_first_is_heads

0.5

In [7]:
assert np.abs(p_both_heads_given_first_is_heads - 1 / 2) < 1e-8

The probability that at least one coin flip comes up heads.

In [8]:
p_at_least_one_heads = float(np.mean(exact_frame['at least 1 heads']))

p_at_least_one_heads

0.75

In [9]:
assert np.abs(p_at_least_one_heads - 3 / 4) < 1e-8

A trickier one: the probability that both flips come come up heads given that at least one of them did. There are 4 possibilities: HH, HT, TH, TT and knowing that at least one flip came up heads eliminates the TT case.

In [10]:
p_both_heads_given_at_least_one_heads = float(np.mean(exact_frame.loc[exact_frame['at least 1 heads'], 'both heads']))

p_both_heads_given_at_least_one_heads

0.3333333333333333

In [11]:
assert np.abs(p_both_heads_given_at_least_one_heads - 1 / 3) < 1e-8

And now the translation of the "Boy Born on Tuesday": the probability that both coin flips come up heads given at least one came up heads on a Tuesday. Notice the answer isn't any of the previous answers, as this variation is in fact different.

In [12]:
at_least_one_heads_on_Tuesday_frame = exact_frame.loc[exact_frame['at least one heads on Tuesday'], :]
p_both_heads_given_at_least_one_heads_born_on_Tuesday = float(at_least_one_heads_on_Tuesday_frame['both heads'].mean())

p_both_heads_given_at_least_one_heads_born_on_Tuesday

0.48148148148148145

In [13]:
numerator = int(at_least_one_heads_on_Tuesday_frame['both heads'].sum())
denominator = int(at_least_one_heads_on_Tuesday_frame.shape[0])
solution = sp.Rational(numerator, denominator)
solution

13/27

In [14]:
float(solution)

0.48148148148148145

In [15]:
assert np.abs(p_both_heads_given_at_least_one_heads_born_on_Tuesday - float(solution)) < 1e-8